In [39]:
from google import genai
from google.genai import types
import pandas as pd
import tiktoken

from qdrant_client.http.models import Filter, FieldCondition, MatchAny

from qdrant_client import QdrantClient
from qdrant_client import models
from qdrant_client.http.models import  VectorParams, Distance, SparseVectorParams, Modifier, PayloadSchemaType, PointStruct, Document, Prefetch, FusionQuery


### Create Qdrant Collection for Hybrid search

In [2]:
qdrant_client = QdrantClient(url="http://localhost:6333")

In [3]:
qdrant_client.create_collection(
    collection_name="Amazon-reviews-collection-01",
    vectors_config={
        "gemini-embedding-001": VectorParams(size=3072, distance=Distance.COSINE)
    }
)

True

In [4]:
qdrant_client.create_payload_index(
    collection_name="Amazon-reviews-collection-01",
    field_name="parent_asin",
    field_schema=PayloadSchemaType.KEYWORD,
)

UpdateResult(operation_id=2, status=<UpdateStatus.COMPLETED: 'completed'>)

### Embedding Functions

In [5]:
import os
gemini_client = genai.Client(api_key=os.getenv("GOOGLE_API_KEY"))
def get_embeddings(text, task_type="SEMANTIC_SIMILARITY", model="gemini-embedding-001"):
    result = gemini_client.models.embed_content(
        model=model,
        contents=text,
        config=types.EmbedContentConfig(task_type=task_type)
    )
    return result.embeddings[0].values

In [6]:
def get_embeddings_batch(text_list, model="gemini-embedding-001", batch_size=100):
    if len(text_list) <= batch_size:
        response = gemini_client.models.embed_content(model=model, contents=text_list, config=types.EmbedContentConfig(task_type="SEMANTIC_SIMILARITY"))
        return [embeddings.values for embeddings in response.embeddings]
    all_embeddings = []
    counter = 1
    for i in range(0, len(text_list), batch_size):
        batch = text_list[i : i + batch_size]
        response = gemini_client.models.embed_content(model=model, contents=batch, config=types.EmbedContentConfig(task_type="SEMANTIC_SIMILARITY"))
        all_embeddings.extend([embeddings.values for embeddings in response.embeddings])
        print(f"Processed {counter * batch_size} out of {len(text_list)}")
        counter += 1
    return all_embeddings

In [7]:
df_reviews = pd.read_json("../../data/Electronics_2022_2023_with_category_ratings_100_sample_1000.jsonl", lines=True)

In [8]:
df_reviews.head()

,rating,title,text,images,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase
0,4,Nixplay 10.1 touch screen digital picture frame,I purchased this digital frame on a treasure t...,[],B096DQF21Z,B0BNXXNBB4,AFZUK3MTBIBEDQOPAK3OATUOUKLA,2022-07-29 06:52:30.702,19,True
1,5,Great so far...,"Speedy delivery, great sound and a great warra...",[],B08H1WNYTR,B0C72D4J46,AFANVB6MPHJTCTFOVIEBKLWZ2GVA,2022-07-05 14:40:48.001,0,True
2,5,Set up is not that easy.,Nice looking set but installation instructions...,[],B09XGX1GMJ,B09XGQQ98K,AHACLF2COQQE2V33ZFXQ7THZOJ2Q,2022-09-21 11:26:42.074,0,True
3,2,Waste of money,"Very unhappy with this keyboard, it would slip...",[],B07899MFZ2,B07L5L22ZL,AGYEAZK4OEYF2MSSTGJ5WNJDVZKA,2018-09-29 22:39:47.708,1,True
4,5,Nice,Work great,[],B09JSMNZRG,B09LTX3SQX,AH67BI7JTOFR35HMZYFVOEHM4CPQ,2023-01-11 21:15:08.805,0,True


In [9]:
len(df_reviews)

122840

In [10]:
### Preprocess title and features

def preprocess_reviews_data(row):
    return f"{row['title']} {(row['text'])}"

In [16]:
### don't know if it will work on  gemini

def count_tokens(row):
    encoding = tiktoken.get_encoding("cl100k_base") 
    return len(encoding.encode(row['preprocessed_data']))



In [17]:
df_reviews['preprocessed_data'] = df_reviews.apply(preprocess_reviews_data, axis=1)
df_reviews['token_count'] = df_reviews.apply(count_tokens, axis=1)

In [18]:
df_reviews.head()

,rating,title,text,images,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase,preprocessed_data,token_count
0,4,Nixplay 10.1 touch screen digital picture frame,I purchased this digital frame on a treasure t...,[],B096DQF21Z,B0BNXXNBB4,AFZUK3MTBIBEDQOPAK3OATUOUKLA,2022-07-29 06:52:30.702,19,True,Nixplay 10.1 touch screen digital picture fram...,255
1,5,Great so far...,"Speedy delivery, great sound and a great warra...",[],B08H1WNYTR,B0C72D4J46,AFANVB6MPHJTCTFOVIEBKLWZ2GVA,2022-07-05 14:40:48.001,0,True,"Great so far... Speedy delivery, great sound a...",27
2,5,Set up is not that easy.,Nice looking set but installation instructions...,[],B09XGX1GMJ,B09XGQQ98K,AHACLF2COQQE2V33ZFXQ7THZOJ2Q,2022-09-21 11:26:42.074,0,True,Set up is not that easy. Nice looking set but ...,18
3,2,Waste of money,"Very unhappy with this keyboard, it would slip...",[],B07899MFZ2,B07L5L22ZL,AGYEAZK4OEYF2MSSTGJ5WNJDVZKA,2018-09-29 22:39:47.708,1,True,Waste of money Very unhappy with this keyboard...,66
4,5,Nice,Work great,[],B09JSMNZRG,B09LTX3SQX,AH67BI7JTOFR35HMZYFVOEHM4CPQ,2023-01-11 21:15:08.805,0,True,Nice Work great,3


In [19]:
df_reviews=df_reviews[df_reviews['token_count']<8192]

In [20]:
len(df_reviews)

122840

In [21]:
total_tokens = df_reviews['token_count'].sum()

In [22]:
total_tokens

np.int64(6506026)

### Embed this text and additional fields for reviews

In [24]:
df_data_to_embed = df_reviews[['preprocessed_data', 'parent_asin']]

In [25]:
data_to_embed_reviews = df_data_to_embed.to_dict(orient = "records")

In [26]:
data_to_embed_reviews

[{'preprocessed_data': "Nixplay 10.1 touch screen digital picture frame I purchased this digital frame on a treasure truck deal - it  works.  You will have to load the app and follow directions to bluetooth transfer pics you choose from your gallery to the frame.  I purchased an additional memory chip - but it turns out there is no place to install the chip so don't waste your time there.  You can set a regular time to 'run' the frame or manually turn it off and/or on.  You can 'shuffle' OR play all your uploaded pics.  You do not get to choose the 'fade' that happens between pics, but you do get to choose how long each pic stays on the frame before moving to the next pic. Picture quality is pretty good - about as good as the pics you download on to it.  It's close to an 8x10 frame.  Not sure why they call it a 'touch screen' cause I can't figure out what you can do by touching the screen - but ok. It comes with charging cord, regular plug in - there appears to be a mini USB port, but 

In [28]:
len(data_to_embed_reviews)

122840

In [29]:
text_to_embed = [item['preprocessed_data'] for item in data_to_embed_reviews]

In [30]:
text_to_embed

["Nixplay 10.1 touch screen digital picture frame I purchased this digital frame on a treasure truck deal - it  works.  You will have to load the app and follow directions to bluetooth transfer pics you choose from your gallery to the frame.  I purchased an additional memory chip - but it turns out there is no place to install the chip so don't waste your time there.  You can set a regular time to 'run' the frame or manually turn it off and/or on.  You can 'shuffle' OR play all your uploaded pics.  You do not get to choose the 'fade' that happens between pics, but you do get to choose how long each pic stays on the frame before moving to the next pic. Picture quality is pretty good - about as good as the pics you download on to it.  It's close to an 8x10 frame.  Not sure why they call it a 'touch screen' cause I can't figure out what you can do by touching the screen - but ok. It comes with charging cord, regular plug in - there appears to be a mini USB port, but I haven't tried it.  H

In [32]:
embeddings = get_embeddings_batch(text_to_embed, batch_size=100)

Processed 100 out of 122840
Processed 200 out of 122840
Processed 300 out of 122840
Processed 400 out of 122840
Processed 500 out of 122840
Processed 600 out of 122840
Processed 700 out of 122840
Processed 800 out of 122840
Processed 900 out of 122840
Processed 1000 out of 122840
Processed 1100 out of 122840
Processed 1200 out of 122840
Processed 1300 out of 122840
Processed 1400 out of 122840
Processed 1500 out of 122840
Processed 1600 out of 122840
Processed 1700 out of 122840
Processed 1800 out of 122840
Processed 1900 out of 122840
Processed 2000 out of 122840
Processed 2100 out of 122840
Processed 2200 out of 122840
Processed 2300 out of 122840
Processed 2400 out of 122840
Processed 2500 out of 122840
Processed 2600 out of 122840
Processed 2700 out of 122840
Processed 2800 out of 122840
Processed 2900 out of 122840
Processed 3000 out of 122840
Processed 3100 out of 122840
Processed 3200 out of 122840
Processed 3300 out of 122840
Processed 3400 out of 122840
Processed 3500 out of 1

In [33]:
len(embeddings)

122840

In [35]:
pointstructs = []
i=1
for embedding, data in zip(embeddings, data_to_embed_reviews):
    pointstructs.append(
        PointStruct(
            id=i,
            vector={
                "gemini-embedding-001": embedding,
            },
            payload=data
        )
    )
    i += 1

In [36]:
pointstructs[0].vector

{'gemini-embedding-001': [-0.005144955,
  -0.020206027,
  0.0023758814,
  -0.061007924,
  -0.0070066606,
  -0.01569244,
  0.013597226,
  0.009579421,
  0.001975097,
  0.005852266,
  -0.011311572,
  -0.019509774,
  -0.020549167,
  0.010739026,
  0.11935063,
  0.00732428,
  -0.008933963,
  -0.011288102,
  0.0028880239,
  0.0037648291,
  -0.007987547,
  0.0045604785,
  -0.025517518,
  -0.014720928,
  -0.009071079,
  -0.022528278,
  0.04917037,
  0.016513312,
  0.028548287,
  -0.010973278,
  -0.0024716333,
  0.020403598,
  0.0040372084,
  0.0068851104,
  -0.002104653,
  0.004421617,
  0.026869645,
  -0.016160453,
  -0.01602107,
  0.029419629,
  -0.013903111,
  0.014549532,
  -0.0007782127,
  0.011384588,
  -0.015504973,
  0.004628719,
  0.00024946893,
  -0.020536471,
  0.020214936,
  0.026762383,
  0.0011473525,
  -0.01368845,
  0.01747721,
  -0.1734683,
  -0.0047676614,
  -0.0016835601,
  -0.0049786204,
  -0.0071396464,
  0.020647671,
  -0.011353411,
  -0.0381832,
  0.029338004,
  -0.0225

In [37]:
len(pointstructs)

122840

In [38]:
batch_size_qdrant = 100
counter = 1
for i in range(0, len(pointstructs), batch_size_qdrant):
    batch = pointstructs[i : i + batch_size_qdrant]
    qdrant_client.upsert(
        collection_name="Amazon-reviews-collection-01",
        points=batch,
        wait=True
    )
    print(f"Processed {counter * batch_size_qdrant} out of {len(pointstructs)}")
    counter += 1

Processed 100 out of 122840
Processed 200 out of 122840
Processed 300 out of 122840
Processed 400 out of 122840
Processed 500 out of 122840
Processed 600 out of 122840
Processed 700 out of 122840
Processed 800 out of 122840
Processed 900 out of 122840
Processed 1000 out of 122840
Processed 1100 out of 122840
Processed 1200 out of 122840
Processed 1300 out of 122840
Processed 1400 out of 122840
Processed 1500 out of 122840
Processed 1600 out of 122840
Processed 1700 out of 122840
Processed 1800 out of 122840
Processed 1900 out of 122840
Processed 2000 out of 122840
Processed 2100 out of 122840
Processed 2200 out of 122840
Processed 2300 out of 122840
Processed 2400 out of 122840
Processed 2500 out of 122840
Processed 2600 out of 122840
Processed 2700 out of 122840
Processed 2800 out of 122840
Processed 2900 out of 122840
Processed 3000 out of 122840
Processed 3100 out of 122840
Processed 3200 out of 122840
Processed 3300 out of 122840
Processed 3400 out of 122840
Processed 3500 out of 1

### A function to run search against reviews

In [42]:
def retrieve_prefiltered_data(query, parent_asins, k=5):
    query_embedding = get_embeddings(query)
    results = qdrant_client.query_points(
            collection_name="Amazon-reviews-collection-01",
            prefetch=[
                Prefetch(
                    query=query_embedding,
                    using="gemini-embedding-001",
                    filter=Filter(
                        must=[
                            FieldCondition(
                                key="parent_asin",
                                match=MatchAny(
                                    any=parent_asins
                                )
                            )
                        ]
                    ),
                    limit=20
                )
            ],
            query=FusionQuery(fusion='rrf'),
            limit=k
    )
    return results
   

In [43]:
reviews =retrieve_prefiltered_data("bad_quality", ["B09Q5TNDHY"])

In [44]:
reviews

QueryResponse(points=[ScoredPoint(id=62005, version=623, score=0.5, payload={'preprocessed_data': 'Its a total waste and the screen came as if it was used before The screen unlike the picture is so small and when i took it out of the box it looks like the watch was used before its not new<br />wouldn’t recommend', 'parent_asin': 'B09Q5TNDHY'}, vector=None, shard_key=None, order_value=None), ScoredPoint(id=80491, version=807, score=0.33333334, payload={'preprocessed_data': "Its stopped working It's not that durable", 'parent_asin': 'B09Q5TNDHY'}, vector=None, shard_key=None, order_value=None), ScoredPoint(id=1606, version=19, score=0.25, payload={'preprocessed_data': 'Lasted less than 4 months Battery life is very short, and now touch screen messed up. When I tried to toch weather, then my message opens, when I tried to touch heart rate, it opens my Training... not worse getting. Wasting money and not available for replacement because they are out of stock.', 'parent_asin': 'B09Q5TNDHY'

In [45]:
reviews =retrieve_prefiltered_data("bad quality", ["B09Q5TNDHY", "B0BHVH4D37"])

In [46]:
reviews

QueryResponse(points=[ScoredPoint(id=74194, version=744, score=0.5, payload={'preprocessed_data': "Bad viewing angle, slow, and poor build quality I was going to use this as a basic tablet but even for this use it's painfully slow. This is the case of you get what you pay for. At this price I recommend just getting a Amazon Fire HD7 or 8.<br /><br />Cons:<br />Screen has a bad viewing angle. Looking at it straight on, you can even see the whole screen.<br />Very slow. I thought quad core and 3gb of RAM would run decently but it's really slow.<br />Bad WiFi. It randomly disconnected from the WiFi during the setup process, so I had to attempt it 4 times.<br /><br />Pros:<br />The tablet is cheap.<br />Natively has google play store and apps.", 'parent_asin': 'B0BHVH4D37'}, vector=None, shard_key=None, order_value=None), ScoredPoint(id=15520, version=158, score=0.33333334, payload={'preprocessed_data': "Poor on so many levels Where to begin... The tablet arrived on time and nicely package

### Define the reviews retrieval Tool

In [47]:
def get_embeddings(text, task_type="SEMANTIC_SIMILARITY", model="gemini-embedding-001"):
    result = gemini_client.models.embed_content(
        model=model,
        contents=text,
        config=types.EmbedContentConfig(task_type=task_type)
    )
    return result.embeddings[0].values

def retrieve_prefiltered_reviews_data(query, parent_asins, k=5):
    query_embedding = get_embeddings(query)
    qdrant_client = QdrantClient(url="http://localhost:6333")
    results = qdrant_client.query_points(
            collection_name="Amazon-reviews-collection-01",
            prefetch=[
                Prefetch(
                    query=query_embedding,
                    using="gemini-embedding-001",
                    filter=Filter(
                        must=[
                            FieldCondition(
                                key="parent_asin",
                                match=MatchAny(
                                    any=parent_asins
                                )
                            )
                        ]
                    ),
                    limit=20
                )
            ],
            query=FusionQuery(fusion='rrf'),
            limit=k
    )
    retrieved_context_ids=[]
    retrieved_context=[]
    similarity_scores=[]

    for result in results.points:
        retrieved_context_ids.append(result.payload["parent_asin"])
        retrieved_context.append(result.payload["preprocessed_data"])
        similarity_scores.append(result.score)

    return {
        "retrieved_context_ids": retrieved_context_ids,
        "retrieved_context": retrieved_context,
        "similarity_scores": similarity_scores,
    }

def process_reviews_context(context):
    formatted_context = ""
    for id, chunk in zip(context["retrieved_context_ids"], context["retrieved_context"]):
        formatted_context += f"- ID: {id}, description: {chunk}\n"
    return formatted_context

def get_formatted_reviews_context(query: str,parent_asins: list[str], top_k: int = 5) -> str:
    """ 
    Get the top k reviews matching a query for a list of prefiltered items

    Args:
        query: The query to get the context for
        parent_asins: The list of parent ASINs to filter the context for
        top_k: The number of context to return

    Returns:
        A string of the top k context chunks with IDs prepending for each chun, each representing a review for a given inventory item.
    """

    retrieved_context = retrieve_prefiltered_reviews_data(query, parent_asins, top_k)

    formatted_context = process_reviews_context(retrieved_context)

    return formatted_context

In [48]:
result = get_formatted_reviews_context("bad quality", ["B09Q5TNDHY", "B0BHVH4D37"], 5)

In [ ]:
print(result)

- ID: B0BHVH4D37, description: Bad viewing angle, slow, and poor build quality I was going to use this as a basic tablet but even for this use it's painfully slow. This is the case of you get what you pay for. At this price I recommend just getting a Amazon Fire HD7 or 8.<br /><br />Cons:<br />Screen has a bad viewing angle. Looking at it straight on, you can even see the whole screen.<br />Very slow. I thought quad core and 3gb of RAM would run decently but it's really slow.<br />Bad WiFi. It randomly disconnected from the WiFi during the setup process, so I had to attempt it 4 times.<br /><br />Pros:<br />The tablet is cheap.<br />Natively has google play store and apps.
- ID: B0BHVH4D37, description: Poor on so many levels Where to begin... The tablet arrived on time and nicely packaged. It was nearly fully charged and did power on, though, initial setup was a pain. It does in fact come with android 12. That's the good.<br /><br />The screen is not good at all. Unless you are lookin

: 